# 🧠 Fine-tuning Qwen2.5-3B bằng QLoRA

Notebook này hướng dẫn bạn cách fine-tune model cho tác vụ tóm tắt cuộc họp.
*Lưu ý: Nếu bạn chưa cài đặt các thư viện cần thiết, hãy mở terminal và cài đặt:* `pip install trl peft bitsandbytes accelerate datasets`

In [1]:
import sys
import os
import logging

# Hiển thị log chi tiết
logging.basicConfig(level=logging.INFO)

# Khai báo thư mục gốc để nạp các modules
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from modules.model_loader import load_model_and_tokenizer
from modules.dataset_utils import load_and_split_dataset
from modules.finetuner import prepare_model_for_lora, get_training_arguments, create_trainer

c:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Nạp Dataset

In [2]:
# Ở đây nếu bạn muốn dùng data v1 thì để là "v1", nếu muốn dùng v2 thì sửa thành "v2"
data_dir = os.path.join(PROJECT_ROOT, "data", "raw", "v3")
dataset_dict = load_and_split_dataset(data_dir, val_size=0.1, test_size=0.1)
train_dataset = dataset_dict['train']
val_dataset = dataset_dict['val']
test_dataset = dataset_dict['test']
print(f"Số lượng tập train: {len(train_dataset)} | val: {len(val_dataset)} | test: {len(test_dataset)}")
print("\nMẫu đầu tiên (chuỗi text đầu vào mô hình):\n")
print(train_dataset[0]['text'])


Số lượng tập train: 680 | val: 85 | test: 86

Mẫu đầu tiên (chuỗi text đầu vào mô hình):

Hãy cập nhật lại báo cáo cuộc họp trước đó bằng cách bổ sung thêm thông tin từ nội dung thảo luận mới dưới đây.

### Đầu vào (Báo cáo cũ & Nội dung thảo luận mới):
# Ra mắt bộ sưu tập Summer Breeze 2024 và chiến lược thời trang bền vững

## I. Nội dung chính

### 1. Mục tiêu cuộc họp
- Chốt mẫu thiết kế và chất liệu cho bộ sưu tập hè "Summer Breeze 2024".
- Thảo luận việc ứng dụng các chất liệu tự nhiên (linen, sợi dứa, tơ chuối) và quy trình sản xuất sạch.
- Xây dựng chiến dịch truyền thông kể chuyện (storytelling) về nguồn gốc sản phẩm.
- Thống nhất chính sách giá, dịch vụ bảo hành trọn đời và chương trình thu cũ đổi mới.

### 2. Các vấn đề đã thảo luận
- Bổ sung tone màu nóng (Peach Fuzz, xanh mint) để bắt kịp xu hướng Pantone 2024.
- Hợp tác với các làng nghề tại Lâm Đồng để cung ứng vải sợi tự nhiên.
- Yêu cầu kỹ thuật may không vắt sổ nhựa và sử dụng nút áo bằng vỏ ốc/gỗ.
- Xử lý độ co rút v

### 2. Tải và cấu hình Base Model (4-bit)
Nếu bạn đã tải model về sẵn trong máy, hãy cung cấp tên thư mục qua biến `model_name`.

In [3]:
# Ví dụ: Nếu bạn đã tải model về thư mục qwen2.5-3b-meeting-summarization/models/Qwen2.5-3B/
# Hãy đổi model_name="Qwen2.5-3B". Nếu không nó sẽ dùng tên mặc định trên Hugging Face

model_name = "Qwen/Qwen2.5-3B" # Thay bằng tên thư mục model local của bạn nếu cần tải offline

model, tokenizer = load_model_and_tokenizer(
    model_name=model_name,
    use_4bit=True,
    torch_dtype="float16",
    device_map="auto"
)

# Gắn LoRA vào Model để bắt đầu quá trình Training
model = prepare_model_for_lora(model)

INFO:modules.model_loader:Đang tải tokenizer: Qwen/Qwen2.5-3B
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B/3aab1f1954e9cc14eb9509a215f9e5ca08227a9b/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B/3aab1f1954e9cc14eb9509a215f9e5ca08227a9b/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-3B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-3B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET

trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827


### 3. Bắt đầu quá trình Huấn Luyện (Training)

In [4]:
output_dir = os.path.join(PROJECT_ROOT, "models", "qwen25-3b-v1")

# Thiết lập tham số
training_args = get_training_arguments(
    output_dir=output_dir, 
    num_train_epochs=5 # <--- BẠN CÓ THỂ CHỈNH SỐ LƯỢNG EPOCH Ở ĐÂY (Vd: 5, 10, v.v.)
)

# Khởi tạo Trainer
trainer = create_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    training_args=training_args,
    max_seq_length=1024 # Độ dài tối đa mỗi mẫu
)

# Chạy train
trainer.train()

INFO:modules.finetuner:Đang khởi tạo SFTTrainer.
Truncating eval dataset: 100%|██████████| 85/85 [00:00<00:00, 5591.00 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
25,1.305792,1.191871
50,1.163964,1.044467
75,0.959351,0.970287
100,0.989904,0.928238
125,0.738281,0.905842
150,0.709996,0.891301
175,0.873385,0.880336
200,0.716572,0.877326
225,0.821604,0.866870
250,0.856957,0.859143


INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B/3aab1f1954e9cc14eb9509a215f9e5ca08227a9b/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B/3aab1f1954e9cc14eb9509a215f9e5ca08227a9b/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B/3aab1f1954e9cc14eb9509a215f9e5ca08227a9b/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B/resolve/main/config.json "HTTP/1.1 307 Temporary Red

TrainOutput(global_step=850, training_loss=0.715001803706674, metrics={'train_runtime': 6461.3155, 'train_samples_per_second': 0.526, 'train_steps_per_second': 0.132, 'total_flos': 5.648186818117632e+16, 'train_loss': 0.715001803706674})

### 4. Lưu Adapter

In [6]:
# Lưu LoRA Weights vào thư mục models/qwen2_5_meeting_lora/
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Đã lưu mô hình Fine-tuned vào: {output_dir}")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B/3aab1f1954e9cc14eb9509a215f9e5ca08227a9b/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B/3aab1f1954e9cc14eb9509a215f9e5ca08227a9b/config.json "HTTP/1.1 200 OK"


Đã lưu mô hình Fine-tuned vào: c:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\models\qwen25-3b-v1
